# Oracle net → real-field best response

Do this **in order**. Do not start training until the gap cell finishes.

1. **Probe the pool** — every opponent must load and play through the training harness (the interface-check that has bitten submissions).
2. **Measure the gap** (~40 games, 32 sims, tournament seats) **before** any updates. Read the per-opponent table: who wipes you, who is close. Inncenta is the one to study.
3. **Best-response self-play** from the oracle clone against mixed 4-player tables: the four real opponents + ASU + fixed agents + frozen past-selves. Reward is net-worth margin. Promote only if the candidate beats the incumbent **against the tournament three**, not against ASU.
4. Track per-opponent WR every generation. Gains vs Inncenta / Alinebidal / Slayer matter; farming Expo or fixed bots does not.

**Upload before you run**

- `DeepRL_Monopoly.tar.gz` of **this working tree** (must include `oracle/field_br.py`). GitHub HEAD is not enough until that file is pushed.
- `hybrid_clone_0000.pt` — the current oracle net. Runs dirs are usually missing from tarballs. Put it at `/content/hybrid_clone_0000.pt`.

Colab sessions die. Gap, collect, and field-eval checkpoint every game; re-run the same cell with `resume=True`. Do not rely on Drive mount.

**Tournament table** (4 players): clone vs `alinebidal-final`, `slayer-v1`, `inncenta-heuristic`. Expo is the fourth real opponent — probed and mixed into training, plus a short smoke eval so it actually loads.

In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys, tarfile, urllib.request

CONTENT = Path(os.environ.get('MONOPOLYZERO_CONTENT', '/content'))
JOB = {
    'commit': '5da929b71bb06c24e1977bc0f8ad9ddaebdfdf0e',  # GitHub fallback only
    'repo': 'ToprakG/DeepRL_Monopoly',
    'clone': '/content/hybrid_clone_0000.pt',
    'gap_games': 40,
    'expo_games': 8,
    'sims': 32,
    'workers': 0,            # 0 = min(4, cpu_count)
    'run_gap': True,         # phase 1 — do this first
    'run_train': True,       # set False to stop after the gap table
    'generations': 3,
    'games_per_generation': 32,
    'promotion_games': 40,   # same 32-sim tournament gate as the gap
    'updates': 1000,
    'batch_size': 256,
    'device': 'auto',
    'seed': 0,
}
job_path = CONTENT / 'oracle-field-br-job.json'
if job_path.exists():
    JOB.update(json.loads(job_path.read_text()))
RUN_DIR = CONTENT / 'oracle-field-br-run'
STATUS_PATH = CONTENT / 'oracle-field-br-status.json'
GAP_DIR = RUN_DIR / 'gap'
print(json.dumps(JOB, indent=2, sort_keys=True))

In [ ]:
import torch
ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
STATUS = {
    **JOB, 'state': 'setup',
    'cpu_count': os.cpu_count(),
    'ram_gib': round(ram_gib, 2),
    'torch': torch.__version__,
    'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
print(json.dumps(STATUS, indent=2))

In [ ]:
repository_archive = CONTENT / 'DeepRL_Monopoly.tar.gz'
if repository_archive.exists():
    with tarfile.open(repository_archive, 'r:gz') as archive:
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / 'DeepRL_Monopoly'
else:
    assert re.fullmatch(r'[0-9a-f]{40}', JOB['commit']), JOB['commit']
    repository_archive = CONTENT / f"DeepRL_Monopoly-{JOB['commit']}.tar.gz"
    urllib.request.urlretrieve(
        f"https://codeload.github.com/{JOB['repo']}/tar.gz/{JOB['commit']}",
        repository_archive,
    )
    with tarfile.open(repository_archive, 'r:gz') as archive:
        roots = {Path(member.name).parts[0] for member in archive.getmembers() if member.name}
        assert len(roots) == 1
        archive.extractall(CONTENT, filter='data')
    REPOSITORY_ROOT = CONTENT / roots.pop()
assert (REPOSITORY_ROOT / 'oracle' / 'field_br.py').is_file(), (
    'oracle/field_br.py missing — upload a tarball of the working tree, not an old GitHub commit'
)
sys.path.insert(0, str(REPOSITORY_ROOT))
os.chdir(REPOSITORY_ROOT)

clone_candidates = [
    Path(JOB['clone']),
    CONTENT / 'hybrid_clone_0000.pt',
    REPOSITORY_ROOT / 'monopoly_bench/runs/oracle_hybrid_bc_new25k/snapshots/hybrid_clone_0000.pt',
]
CLONE = next((path for path in clone_candidates if path.is_file()), None)
assert CLONE is not None, 'Upload hybrid_clone_0000.pt to /content/hybrid_clone_0000.pt'
print('repo', REPOSITORY_ROOT)
print('clone', CLONE, 'bytes', CLONE.stat().st_size)

## 2. Probe the real-field opponent pool

Loads Alinebidal, Slayer, Inncenta, Expo, ASU, and the three fixed agents through the **same** `choose_action(game, player_id, decision_seed)` contract the training loop uses, then plays a short game. If this fails, stop — do not build a loop around a broken adapter.

In [ ]:
STATUS['state'] = 'probe'
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
probe_out = RUN_DIR / 'reports' / 'probe.json'
probe_out.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, '-m', 'oracle.field_br', 'probe',
    '--output', str(probe_out),
    '--max-rounds', '8',
]
print(' '.join(cmd), flush=True)
proc = subprocess.run(cmd, cwd=str(REPOSITORY_ROOT), check=False)
probe = json.loads(probe_out.read_text()) if probe_out.exists() else {}
STATUS['probe'] = {'ok': probe.get('ok'), 'failed': probe.get('failed'), 'returncode': proc.returncode}
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
print(json.dumps({'ok': probe.get('ok'), 'failed': probe.get('failed')}, indent=2))
for row in probe.get('reports') or []:
    short = row.get('short_game') or {}
    print(
        f"  {row['policy_id']}: opening={'ok' if row.get('opening_ok') else 'FAIL'} "
        f"short crashes={short.get('crashes')} decisions={short.get('decisions')} "
        f"error={row.get('error') or short.get('error')}"
    )
if proc.returncode != 0 or not probe.get('ok'):
    raise SystemExit('pool probe failed — fix the adapter before gap/train')
print('PROBE OK')

## 1. Gap vs the actual field (do this before training)

Current **oracle net** at **32 sims**, tournament 4-player table, ~40 seat-balanced games.

You already know the headline is roughly 18%. This cell is the **per-opponent** breakdown: which agents wipe you, which are close, net-worth margin vs each. Inncenta at 50%+ on *their* tables is the signal for what self-play has to fix (trades vs building tempo).

A short Expo smoke (default 8 games) is included so all four real opponents actually run. Checkpointed per game — re-run this cell if Colab dies.

In [ ]:
if not JOB['run_gap']:
    print('run_gap=False — skipping')
else:
    STATUS['state'] = 'gap'
    STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
    GAP_DIR.mkdir(parents=True, exist_ok=True)
    gap_json = GAP_DIR / 'gap.json'
    cmd = [
        sys.executable, '-m', 'oracle.field_br', 'gap',
        '--clone', str(CLONE),
        '--games', str(JOB['gap_games']),
        '--expo-games', str(JOB['expo_games']),
        '--sims', str(JOB['sims']),
        '--seed', str(JOB['seed']),
        '--workers', str(JOB['workers']),
        '--checkpoint-dir', str(GAP_DIR),
        '--resume',
        '--output', str(gap_json),
    ]
    print(' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(REPOSITORY_ROOT), check=False)
    STATUS['gap_returncode'] = proc.returncode
    STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
    if proc.returncode != 0:
        raise SystemExit(proc.returncode)
print('gap done')

In [ ]:
gap_json = GAP_DIR / 'gap.json'
assert gap_json.is_file(), 'gap.json missing — run the gap cell'
gap = json.loads(gap_json.read_text())
tourney = gap['tournament']
print('clone', gap['clone'])
print(f"TOURNAMENT  clone WR={tourney['learner_win_rate']:.3f} "
      f"({tourney['learner_wins']}/{tourney['completed']})  "
      f"Wilson {tourney['wilson_95']}")
print('who won:', tourney.get('winner_counts'))
print(f"bankruptcies={tourney.get('bankruptcies')} crashes={tourney.get('crashes')}")
print()
print(f"{'opponent':<24} {'their WR':>8} {'wins':>8} {'our gap':>8} {'NW margin':>12}  note")
print('-' * 78)
rows = []
for name, row in (tourney.get('opponents') or {}).items():
    their = row.get('win_rate')
    gap_pp = row.get('rate_gap')
    mean = row.get('net_worth_margin_mean')
    se = row.get('net_worth_margin_se')
    note = ''
    if their is not None and their >= 0.40:
        note = 'WIPES YOU' if their >= 0.50 else 'strong'
    if name == 'inncenta-heuristic':
        note = (note + '  study this').strip()
    print(
        f"{name:<24} {0 if their is None else 100*their:7.1f}% "
        f"{row.get('wins')}/{row.get('games'):<6} "
        f"{'' if gap_pp is None else f'{100*gap_pp:+.1f}%':>8} "
        f"{'' if mean is None else f'{mean:.0f}':>12}  {note}"
    )
    rows.append((name, 0.0 if their is None else their))

expo = gap.get('expo')
if expo:
    print()
    print(f"EXPO SMOKE  clone WR={expo['learner_win_rate']:.3f} "
          f"({expo['learner_wins']}/{expo['completed']})")
    for name, row in (expo.get('opponents') or {}).items():
        print(f"  vs {name}: their WR={row.get('win_rate')}")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt and rows:
    names, rates = zip(*rows)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    colors = ['#c0392b' if r >= 0.4 else '#2980b9' for r in rates]
    ax.bar(names, [100 * r for r in rates], color=colors)
    ax.axhline(100 * tourney['learner_win_rate'], color='black', ls='--', label='clone WR')
    ax.set_ylabel('opponent win rate %')
    ax.set_title('Gap vs tournament field (32 sims)')
    ax.tick_params(axis='x', rotation=20)
    ax.legend()
    fig.tight_layout()
    fig.savefig(GAP_DIR / 'per_opponent.png', dpi=120)
    plt.show()
print()
print('Read this before training. If one opponent owns the table, that behavior is the BR target.')

## 3–4. Best-response from the oracle net

Warm-start = the clone you just measured. Training tables mix:

- 50% exact tournament three (Alinebidal / Slayer / Inncenta)
- 25% three-of-four real opponents (Expo enters here)
- 25% mixed robustness (ASU + fixed-a/b/c + frozen past-selves)

Outcomes are **net-worth margin**, not one-hot winners. Every generation is gated at 32 sims, tournament conditions, vs the **real field**. Promote only if field WR beats the current agent (the gap number, not ASU).

Lean into exploiting this exact field — that is the scoring — but keep past-selves/ASU/fixed in the pool so an opponent swap before submission does not wipe you. Watch Inncenta / Alinebidal / Slayer WR each generation; ignore a bump that is only Expo/fixed.

Set `JOB['run_train']=False` and stop here if you only wanted the gap.

In [ ]:
if not JOB['run_train']:
    print('run_train=False — stop after the gap. Flip the flag and re-run this cell to train.')
else:
    STATUS['state'] = 'train'
    STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
    gap_json = GAP_DIR / 'gap.json'
    cmd = [
        sys.executable, '-m', 'oracle.field_br', 'train',
        '--clone', str(CLONE),
        '--run-dir', str(RUN_DIR),
        '--generations', str(JOB['generations']),
        '--games-per-generation', str(JOB['games_per_generation']),
        '--promotion-games', str(JOB['promotion_games']),
        '--updates', str(JOB['updates']),
        '--batch-size', str(JOB['batch_size']),
        '--sims', str(JOB['sims']),
        '--seed', str(JOB['seed']),
        '--workers', str(JOB['workers']),
        '--device', str(JOB['device']),
    ]
    if gap_json.is_file():
        cmd.extend(['--gap-json', str(gap_json)])
    print(' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(REPOSITORY_ROOT), check=False)
    STATUS['train_returncode'] = proc.returncode
    STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
    if proc.returncode != 0:
        raise SystemExit(proc.returncode)
print('train loop finished or resumed-complete')

In [ ]:
history_path = RUN_DIR / 'reports' / 'field_history.json'
status_path = RUN_DIR / 'status.json'
if status_path.exists():
    train_status = json.loads(status_path.read_text())
    print('incumbent', train_status.get('incumbent'))
    print('incumbent field WR', train_status.get('incumbent_field_wr'))
    print('promotions', train_status.get('promotions'))
history = json.loads(history_path.read_text()) if history_path.exists() else []
print()
print(f"{'gen':>4} {'kind':<10} {'clone WR':>8} {'promote':>8}  strong opponents (their WR)")
print('-' * 88)
for row in history:
    field = row.get('field') or row
    wr = field.get('learner_win_rate') or field.get('candidate_field_wr')
    strong = (row.get('field') or {}).get('strong_opponents') or field.get('strong_opponents') or {}
    bits = []
    for name in ('alinebidal-final', 'slayer-v1', 'inncenta-heuristic'):
        rec = strong.get(name) or {}
        rate = rec.get('win_rate')
        if rate is not None:
            bits.append(f"{name.split('-')[0]} {100*rate:.0f}%")
    print(
        f"{row.get('generation', '?'):>4} {row.get('kind', ''):<10} "
        f"{'' if wr is None else f'{100*wr:.1f}%':>8} "
        f"{str(row.get('promoted', '')):>8}  "
        + ', '.join(bits)
    )

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
gens, clone_wr, inn, ali, sla = [], [], [], [], []
for row in history:
    field = row.get('field') or row
    wr = field.get('learner_win_rate')
    if wr is None:
        continue
    gens.append(row.get('generation', len(gens)))
    clone_wr.append(100 * wr)
    strong = field.get('strong_opponents') or {}
    inn.append(100 * (strong.get('inncenta-heuristic') or {}).get('win_rate') or 0)
    ali.append(100 * (strong.get('alinebidal-final') or {}).get('win_rate') or 0)
    sla.append(100 * (strong.get('slayer-v1') or {}).get('win_rate') or 0)
if plt and gens:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(gens, clone_wr, 'k-o', label='clone WR (want up)')
    ax.plot(gens, ali, label='Alinebidal their WR (want down)')
    ax.plot(gens, sla, label='Slayer their WR (want down)')
    ax.plot(gens, inn, label='Inncenta their WR (want down)')
    ax.set_xlabel('generation')
    ax.set_ylabel('win rate %')
    ax.set_title('Field gate every generation — watch the strong three, not Expo')
    ax.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'reports' / 'field_history.png', dpi=120)
    plt.show()

## Export

Download `oracle-field-br-result.tar.gz`. It includes gap tables, per-generation field reports, promoted snapshots, and replay. Re-upload it on the next Colab VM and extract over `/content/oracle-field-br-run` to resume.

In [ ]:
STATUS['state'] = 'export'
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')
out = CONTENT / 'oracle-field-br-result.tar.gz'
with tarfile.open(out, 'w:gz') as archive:
    if RUN_DIR.exists():
        archive.add(RUN_DIR, arcname='run')
    archive.add(STATUS_PATH, arcname=STATUS_PATH.name)
print('wrote', out, out.stat().st_size)
STATUS['state'] = 'done'
STATUS_PATH.write_text(json.dumps(STATUS, indent=2) + '\n')